In [1]:
import pandas as pd

combined_df = pd.read_csv("dataset_preprocessed.csv")

In [2]:
dist = pd.crosstab(combined_df["pr_type"], combined_df["AI_flag"], normalize="columns")

dist

AI_flag,0,1
pr_type,,
bug_fix,0.657093,0.729206
documentation,0.013594,0.030955
feature,0.167069,0.054584
maintenance,0.046700,0.104679
other,0.068625,0.004726
refactor,0.023898,0.002836
test,0.023021,0.073015


In [3]:
dist["difference"] = dist[1] - dist[0]

dist.sort_values(by="difference", ascending=False)

AI_flag,0,1,difference
pr_type,,,
bug_fix,0.657093,0.729206,0.072113
maintenance,0.046700,0.104679,0.057978
test,0.023021,0.073015,0.049994
documentation,0.013594,0.030955,0.017361
refactor,0.023898,0.002836,-0.021063
other,0.068625,0.004726,-0.063899
feature,0.167069,0.054584,-0.112485


In [4]:
from scipy.stats import chi2_contingency

table = pd.crosstab(combined_df["pr_type"], combined_df["AI_flag"])

chi2_contingency(table)

Chi2ContingencyResult(statistic=np.float64(815.4780425749491), pvalue=np.float64(6.967192394281226e-173), dof=6, expected_freq=array([[3155.30114864, 2927.69885136],
       [ 100.1106562 ,   92.8893438 ],
       [ 515.07710679,  477.92289321],
       [ 340.27248948,  315.72751052],
       [ 172.72978506,  160.27021494],
       [  62.76367565,   58.23632435],
       [ 214.74513818,  199.25486182]]))

In [5]:
import numpy as np

chi2, p, dof, expected = chi2_contingency(table)
n = table.sum().sum()

cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))

cramers_v

np.float64(0.30453527088855464)

In [6]:
repo_summary = []

for repo in combined_df["repo"].unique():
    subset = combined_df[combined_df["repo"] == repo]
    
    table = pd.crosstab(subset["pr_type"], subset["AI_flag"], normalize="columns")
    
    if "bug_fix" in table.index:
        diff = table.loc["bug_fix", 1] - table.loc["bug_fix", 0]
        repo_summary.append((repo, diff))

repo_summary_df = pd.DataFrame(repo_summary, columns=["repo", "bug_fix_diff"])

repo_summary_df

,repo,bug_fix_diff
0,liam-hq/liam,0.141466
1,glaredb/glaredb,0.518733
2,promptfoo/promptfoo,0.255404
3,antiwork/gumroad,0.015518
4,prebid/prebid.js,0.368834
5,mlflow/mlflow,0.017645
6,pdfme/pdfme,0.094135
7,yamadashy/repomix,0.367293
8,featureform/enrichmcp,0.163134
9,microsoft/testfx,-0.396517


In [7]:
(repo_summary_df["bug_fix_diff"] > 0).sum(), len(repo_summary_df)

(np.int64(9), 12)